# Beacon Heuristics — Reproducible Evaluation

This notebook replicates the Beacon browser extension's heuristic engine in Python
and evaluates it against a labeled dataset of phishing and legitimate URLs.

**Two evaluation phases:**
- **Phase 1 — URL-only:** Runs URL heuristics on the full dataset (no HTTP fetching required). Fast, reliable, works even for URLs that are offline.
- **Phase 2 — Full pipeline:** Fetches live pages and runs all three modules (URL + content + link heuristics). More complete but depends on pages being accessible.

**Heuristic modules replicated from TypeScript:**
- `urlHeuristics.ts` → `analyze_url(url)` — 8 rules (6 Tier 1 + 2 Tier 2)
- `contentHeuristics.ts` → `analyze_content(page_data)` — 7 rules including 5 DOM-level rules
- `linkHeuristics.ts` → `analyze_links(links, current_url)`
- `content.ts:combineResults` → `combine_results(url_r, content_r, link_r)` — URL veto + trusted aggregator bypass

**URL rules (urlHeuristics.ts):**

| Rule | Tier | Weight | Signal |
|---|---|---|---|
| `isdomainip` | 1 | 10 | Raw IPv4 address as hostname |
| `hasobfuscation` | 1 | 8 | `@` credential injection or encoded hostname |
| `urlLengthHard` | 1 | 7 | Path > 144 chars AND domain is also suspicious |
| `brandSubdomainSpoofing` | 1 | 5 | Known brand name in non-brand hostname |
| `freeHostingFinancialSubdomain` | 1 | 5 | Financial keyword in Vercel/Netlify/GitHub Pages subdomain |
| `suspiciousTld` | 1 | 5 | `.icu` `.tk` `.ml` `.ga` `.cf` `.gq` `.cfd` `.cyou` |
| `urlLengthWithComplexity` | 2 | 4 | URL > 75 chars AND (hyphens ≥ 3 OR path encoding ≥ 3) |
| `gibberishDomainLabel` | 2 | 3 | 4+ char domain label with zero vowels |

**Scoring:** Safety score 0–10. Higher = safer. 7–10 = safe, 4–6 = uncertain, 0–3 = scam.

## 1. Setup

In [ ]:
!pip install -q requests beautifulsoup4 scikit-learn pandas matplotlib seaborn

In [ ]:
import re
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from urllib.parse import urlparse, unquote
from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor, as_completed
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# ── Configuration ────────────────────────────────────────────────────────────
DATASET_PATH   = "dataset/urls.csv"   # path relative to this notebook
FETCH_TIMEOUT  = 10                    # seconds per page fetch
MAX_WORKERS    = 5                     # parallel fetch threads
MAX_TEXT_CHARS = 5000                  # mirrors content.ts cap
MAX_LINKS      = 100                   # mirrors content.ts cap

print("Setup complete.")

## 2. Heuristics Engine — Python Replication

Direct translation of the TypeScript heuristics. Logic, thresholds, and scoring are identical to the extension.

In [ ]:
# ── Shared: Score → Verdict ───────────────────────────────────────────────────
# Safety score: 0–10. Higher = safer.
# Mirrors toVerdict() in contentHeuristics.ts.

def to_verdict(safety_score: float) -> str:
    if safety_score >= 7:
        return "safe"
    if safety_score >= 4:
        return "uncertain"
    return "scam"

# ── Shared: URL helpers ───────────────────────────────────────────────────────

def parse_hostname(url: str) -> str:
    try:
        return (urlparse(url).hostname or "").lower()
    except Exception:
        return ""

def extract_base_domain(hostname: str) -> str:
    parts = hostname.split(".")
    return ".".join(parts[-2:]) if len(parts) >= 2 else hostname

def extract_domain(url: str) -> str:
    try:
        url_with_proto = url if url.startswith("http") else f"http://{url}"
        return urlparse(url_with_proto).hostname or ""
    except Exception:
        m = re.match(r"(?:https?://)?(?:www\.)?([^/?#]+)", url)
        return m.group(1) if m else ""

print("Shared utilities loaded.")

In [ ]:
# ── URL Heuristics ────────────────────────────────────────────────────────────
# Replicates urlHeuristics.ts

URL_LENGTH_HARD    = 144   # 99th percentile of phishing URL length in PhiUSIIL
URL_LENGTH_SOFT    = 75    # phishing 75th percentile — used in compound rules
PERCENT_ENCODED_MIN = 3
HOST_HYPHENS_MIN   = 3

def _check_isdomainip(url: str) -> dict:
    hostname = parse_hostname(url)
    is_ip = bool(re.match(r"^(\d{1,3}\.){3}\d{1,3}$", hostname))
    return {
        "rule": "isdomainip", "tier": 1, "weight": 10,
        "triggered": is_ip,
        "finding": f"URL uses a raw IP address ({hostname}) instead of a domain name" if is_ip else ""
    }

def _check_hasobfuscation(url: str) -> dict:
    try:
        parsed = urlparse(url)
        if parsed.username or parsed.password:
            return {
                "rule": "hasobfuscation", "tier": 1, "weight": 8, "triggered": True,
                "finding": f"URL contains '@' credential-injection — actual host is '{parsed.hostname}'"
            }
        if re.search(r"%[0-9a-fA-F]{2}", parsed.hostname or ""):
            return {
                "rule": "hasobfuscation", "tier": 1, "weight": 8, "triggered": True,
                "finding": f"URL hostname contains percent-encoded characters: {parsed.hostname}"
            }
    except Exception:
        pass
    return {"rule": "hasobfuscation", "tier": 1, "weight": 8, "triggered": False, "finding": ""}

def _check_url_length_hard(url: str) -> dict:
    triggered = len(url) > URL_LENGTH_HARD
    return {
        "rule": "urlLengthHard", "tier": 1, "weight": 7,
        "triggered": triggered,
        "finding": f"URL is {len(url)} chars, exceeding the {URL_LENGTH_HARD}-char phishing threshold" if triggered else ""
    }

def _check_url_length_complexity(url: str) -> dict:
    if len(url) <= URL_LENGTH_SOFT:
        return {"rule": "urlLengthWithComplexity", "tier": 2, "weight": 4, "triggered": False, "finding": ""}
    hostname = parse_hostname(url)
    url_without_query = url.split("?")[0]  # exclude query string — encoded params are normal
    pct_encoded = len(re.findall(r"%[0-9a-fA-F]{2}", url_without_query))
    host_hyphens = hostname.count("-")
    triggered = pct_encoded >= PERCENT_ENCODED_MIN or host_hyphens >= HOST_HYPHENS_MIN
    return {
        "rule": "urlLengthWithComplexity", "tier": 2, "weight": 4,
        "triggered": triggered,
        "finding": (
            f"Long URL ({len(url)} chars) with suspicious structure — "
            f"{pct_encoded} %-encoded sequences in path, {host_hyphens} hyphens in hostname"
        ) if triggered else ""
    }

URL_RULES = [_check_isdomainip, _check_hasobfuscation, _check_url_length_hard, _check_url_length_complexity]

def analyze_url(url: str) -> dict:
    findings, fired_rules = [], []
    threat_score = 0
    for check_fn in URL_RULES:
        result = check_fn(url)
        if result["triggered"]:
            threat_score += result["weight"]
            findings.append(f"[Tier {result['tier']}] {result['finding']}")
            fired_rules.append(result["rule"])
    threat_score = min(threat_score, 10)
    safety_score = 10 - threat_score
    return {
        "score": safety_score, "verdict": to_verdict(safety_score),
        "findings": findings, "fired_rules": fired_rules, "source": "url"
    }

print("URL heuristics loaded.")

In [ ]:
# ── Content Heuristics ────────────────────────────────────────────────────────
# Replicates contentHeuristics.ts

SPARSE_TEXT_MAX = 200

SCAM_PHRASES = [
    "you have won", "you've won", "congratulations, you won",
    "claim your prize", "click here to claim", "urgent action required",
    "act now", "limited time offer", "exclusive deal", "send a wire transfer",
    "you are the lucky winner", "congratulations you are our winner",
]

def analyze_content(page_data: dict) -> dict:
    findings = []
    threat_score = 0

    # sparsityNoMeta rule: proxy for largestlinelength + lineofcode (EDA Finding 3.8)
    text_len = len((page_data.get("textContent") or "").strip())
    has_meta = len((page_data.get("metaDescription") or "").strip()) > 0
    if text_len < SPARSE_TEXT_MAX and not has_meta:
        threat_score += 3
        findings.append(f"Sparse page: {text_len} chars of body text and no meta description")

    # Scam phrase detection
    title = (page_data.get("title") or "").lower()
    meta  = (page_data.get("metaDescription") or "").lower()
    body  = (page_data.get("textContent") or "").lower()

    for phrase in SCAM_PHRASES:
        if phrase in title:
            threat_score += 3
            findings.append(f'Scam phrase in title: "{phrase}"')
        if phrase in meta:
            threat_score += 3
            findings.append(f'Scam phrase in meta: "{phrase}"')
        if phrase in body:
            threat_score += 2
            findings.append(f'Scam phrase in body: "{phrase}"')

    threat_score = min(threat_score, 10)
    safety_score = 10 - threat_score
    return {
        "score": safety_score, "verdict": to_verdict(safety_score),
        "findings": findings, "source": "content"
    }

print("Content heuristics loaded.")

In [ ]:
# ── Link Heuristics ───────────────────────────────────────────────────────────
# Replicates linkHeuristics.ts

COMMON_TYPOSQUATTING = {
    "google":    ["googl", "gogle", "goog1e", "g00gle", "g0ogle"],
    "amazon":    ["amazo", "amaz0n", "am4zon"],
    "facebook":  ["facebok", "faceb00k", "f4cebook", "faceboo"],
    "paypal":    ["p4yp4l", "paypa1", "p@ypal"],
    "apple":     ["4pple", "@pple", "appie"],
    "microsoft": ["microsof", "micr0soft", "m1cr0s0ft"],
    "twitter":   ["twiter", "tw1tter"],
    "instagram": ["instag4m", "insta9ram", "inst4gram"],
    "github":    ["g1thub", "gihub"],
    "linkedin":  ["link3d1n", "1inkedin"],
}
HOMOGLYPH_MAP = {"0": "o", "1": "l", "3": "e", "4": "a", "5": "s", "7": "t", "8": "b", "9": "g"}
RARE_TLDS = {"tk", "ml", "ga", "cf", "gq", "xyz", "top", "download", "review"}

BRAND_NAMES = set(COMMON_TYPOSQUATTING.keys())

def _lnk_check_excessive_length(url):
    d = extract_domain(url)
    return (True, f"Domain is {len(d)} chars (>45)") if len(d) > 45 else (False, "")

def _lnk_check_insufficient_length(url):
    d = extract_domain(url)
    parts = d.split(".")
    base = ".".join(parts[:-1])
    return (True, f"Domain base is only {len(base)} chars (<6)") if 0 < len(base) < 6 else (False, "")

def _lnk_check_ip(url):
    if re.match(r"^(?:https?://)?(?:\d{1,3}\.){3}\d{1,3}", url):
        return (True, "URL uses IPv4 address instead of domain name")
    if re.match(r"^(?:https?://)?\[?([0-9a-fA-F]{0,4}:){2,7}[0-9a-fA-F]{0,4}\]?", url):
        return (True, "URL uses IPv6 address instead of domain name")
    return (False, "")

def _lnk_check_https(url):
    lower = url.lower()
    if not lower.startswith("https://"):
        reason = "URL uses HTTP instead of HTTPS" if lower.startswith("http://") else "URL lacks HTTPS"
        return (True, reason)
    return (False, "")

def _lnk_check_at_symbol(url):
    if "@" in url:
        parts = url.split("@")
        return (True, f"URL contains @ symbol (credential injection risk). Before @: '{parts[0]}'")
    return (False, "")

def _lnk_check_zero_day(url):
    domain = extract_domain(url)
    hyphen_count = domain.count("-")
    parts = domain.split(".")
    domain_name = ".".join(parts[:-1])
    tld = parts[-1].lower() if parts else ""
    if hyphen_count > 2:
        return (True, f"Excessive hyphens in domain ({hyphen_count}), typical of newly registered domains")
    if len(domain_name) > 5 and re.match(r"^[bcdfghjklmnpqrstvwxyz-]+$", domain_name):
        return (True, "Domain pattern suggests randomly generated domain (mostly consonants)")
    if tld in RARE_TLDS:
        return (True, f"Rare/suspicious TLD (.{tld}), commonly used for phishing")
    return (False, "")

def _lnk_check_typosquatting(url):
    domain = extract_domain(url).lower()
    reasons = []
    for brand, misspellings in COMMON_TYPOSQUATTING.items():
        for m in misspellings:
            if m in domain:
                reasons.append(f'Possible typosquatting: "{m}" resembles "{brand}"')
    for num, letter in HOMOGLYPH_MAP.items():
        if num in domain:
            replaced = domain.replace(num, letter)
            for brand in BRAND_NAMES:
                if brand in replaced:
                    reasons.append(f'Homoglyph: "{num}" instead of "{letter}" (resembles "{brand}")')
    if re.search(r"[0-9]{2,}", domain):
        seqs = re.findall(r"[0-9]{2,}", domain)
        reasons.append(f"Multiple consecutive numbers: {', '.join(seqs)}")
    return (len(reasons) > 0, "; ".join(reasons))

def _lnk_check_favicon(url, link_text=""):
    domain = extract_domain(url).lower()
    for brand in BRAND_NAMES:
        if brand not in domain and brand in link_text.lower():
            return (True, f'Favicon mismatch: link text suggests "{brand}" but domain is "{domain}"')
    return (False, "")

LINK_CHECKS = [
    _lnk_check_excessive_length, _lnk_check_insufficient_length, _lnk_check_ip,
    _lnk_check_https, _lnk_check_at_symbol, _lnk_check_zero_day, _lnk_check_favicon,
]

def analyze_link(link: dict) -> dict:
    findings = []
    suspicion = 0
    href = link.get("href", "")
    text = link.get("text", "")
    for check_fn in LINK_CHECKS:
        args = [href, text] if check_fn == _lnk_check_favicon else [href]
        flagged, reason = check_fn(*args)
        if flagged:
            findings.append(f"⚠️ {reason}")
            suspicion += 1
    flagged, reason = _lnk_check_typosquatting(href)  # counts as one flag even with multiple reasons
    if flagged:
        findings.append(f"⚠️ {reason}")
        suspicion += 1
    safety_score = 10 - round((suspicion / 8) * 10)
    verdict = "safe" if suspicion == 0 else ("scam" if suspicion > 4 else "uncertain")
    return {"score": safety_score, "verdict": verdict, "findings": findings, "suspicion": suspicion}

def _extract_claimed_domain(link_text: str):
    text = link_text.strip().lower()
    common_tlds = [".com", ".net", ".org", ".io", ".co", ".uk", ".de", ".jp", ".fr"]
    if not any(text.endswith(t) or t + "/" in text for t in common_tlds):
        return None
    try:
        return urlparse(text).hostname.lower()
    except Exception:
        try:
            return urlparse("http://" + text).hostname.lower()
        except Exception:
            return None

def analyze_links(links: list, current_url: str) -> dict:
    if not links:
        return {"score": 10, "verdict": "safe", "findings": [], "source": "url",
                "mismatch_count": 0, "flagged_link_count": 0}

    findings = []
    threat_score = 0
    current_base = extract_base_domain(parse_hostname(current_url))

    # (A) Mismatched link detection
    mismatch_count = 0
    for link in links:
        claimed = _extract_claimed_domain(link.get("text", ""))
        if not claimed:
            continue
        href_host = parse_hostname(link.get("href", ""))
        if not href_host:
            continue
        if extract_base_domain(href_host) == current_base:
            continue
        if extract_base_domain(claimed) != extract_base_domain(href_host):
            mismatch_count += 1
            findings.append(f"Mismatched link: text claims '{claimed}' but href goes to '{href_host}'")
    threat_score += min(mismatch_count * 4, 8)

    # (B) Per-link URL heuristics (3+ flags = seriously flagged)
    link_results = [analyze_link(lnk) for lnk in links]
    flagged_links = [r for r in link_results if r["score"] <= 6]
    threat_score += min(len(flagged_links), 3)

    for i, result in enumerate(link_results):
        if result["score"] <= 7 and result["findings"]:
            label = links[i].get("text") or links[i].get("href", "")
            findings.append(f"Link ({label[:60]}): {'; '.join(result['findings'][:2])}")

    threat_score = min(threat_score, 10)
    safety_score = 10 - threat_score
    return {
        "score": safety_score, "verdict": to_verdict(safety_score),
        "findings": findings, "source": "url",
        "mismatch_count": mismatch_count, "flagged_link_count": len(flagged_links)
    }

print("Link heuristics loaded.")

In [ ]:
# ── Combined Pipeline ─────────────────────────────────────────────────────────
# Replicates combineResults() in content.ts

def combine_results(url_result: dict, content_result: dict, link_result: dict) -> dict:
    # Each module returns a safety score. Convert to threat, sum, re-invert.
    combined_threat = min(10,
        (10 - url_result["score"]) +
        (10 - content_result["score"]) +
        (10 - link_result["score"])
    )
    safety_score = 10 - combined_threat
    all_findings = url_result["findings"] + content_result["findings"] + link_result["findings"]
    return {
        "score": safety_score,
        "verdict": to_verdict(safety_score),
        "findings": all_findings,
        "source": "combined",
        "url_score": url_result["score"],
        "content_score": content_result["score"],
        "link_score": link_result["score"],
        "fired_url_rules": url_result.get("fired_rules", []),
    }

# Safe defaults for when content can't be fetched
EMPTY_CONTENT = {"score": 10, "verdict": "safe", "findings": [], "source": "content"}
EMPTY_LINKS   = {"score": 10, "verdict": "safe", "findings": [], "source": "url",
                 "mismatch_count": 0, "flagged_link_count": 0}

print("Combined pipeline loaded.")

## 3. Page Fetching

For Phase 2 (full pipeline), we need to fetch the actual page content. Some phishing pages may be offline — the pipeline falls back to URL-only results for those.

In [ ]:
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/120.0.0.0 Safari/537.36"
}

def fetch_page_data(url: str, timeout: int = FETCH_TIMEOUT) -> dict:
    """Fetch a URL and extract the same data as content.ts does in the browser."""
    base = {"url": url, "title": "", "metaDescription": "", "textContent": "",
            "links": [], "fetch_status": "pending"}
    try:
        resp = requests.get(url, timeout=timeout, headers=HEADERS, allow_redirects=True)
        soup = BeautifulSoup(resp.text, "html.parser")

        title_tag = soup.find("title")
        base["title"] = title_tag.get_text(strip=True) if title_tag else ""

        meta_tag = soup.find("meta", {"name": "description"})
        base["metaDescription"] = meta_tag.get("content", "") if meta_tag else ""

        # Prefer semantic content area (mirrors content.ts)
        main = soup.find("main") or soup.find("article") or soup.find(attrs={"role": "main"})
        text_source = main if main else soup.body
        raw_text = text_source.get_text(separator=" ", strip=True) if text_source else ""
        base["textContent"] = raw_text[:MAX_TEXT_CHARS]

        # Extract first MAX_LINKS http/https links with visible text
        links = []
        for a in soup.find_all("a", href=True):
            href = a.get("href", "")
            text = a.get_text(strip=True)
            if text and (href.startswith("http://") or href.startswith("https://")):
                links.append({"text": text, "href": href})
            if len(links) >= MAX_LINKS:
                break
        base["links"] = links
        base["fetch_status"] = f"ok:{resp.status_code}"

    except requests.exceptions.Timeout:
        base["fetch_status"] = "timeout"
    except requests.exceptions.ConnectionError:
        base["fetch_status"] = "connection_error"
    except Exception as e:
        base["fetch_status"] = f"error:{type(e).__name__}"

    return base


def fetch_all(urls: list, max_workers: int = MAX_WORKERS) -> dict:
    """Fetch multiple URLs in parallel. Returns {url: page_data}."""
    results = {}
    with ThreadPoolExecutor(max_workers=max_workers) as pool:
        future_to_url = {pool.submit(fetch_page_data, url): url for url in urls}
        for i, future in enumerate(as_completed(future_to_url), 1):
            url = future_to_url[future]
            results[url] = future.result()
            status = results[url]["fetch_status"]
            print(f"  [{i}/{len(urls)}] {status:20s}  {url[:70]}")
    return results

print("Page fetcher loaded.")

## 4. Load Dataset

In [ ]:
df = pd.read_csv(DATASET_PATH)
print(f"Loaded {len(df)} URLs: {df['label'].value_counts().to_dict()}")
df.head()

## 5. Phase 1 — URL-Only Evaluation

Runs only `analyze_url()` against each URL. No HTTP fetching — works even for offline phishing pages.

In [ ]:
print("Running URL heuristics on all URLs...")
url_rows = []
for _, row in df.iterrows():
    result = analyze_url(row["url"])
    url_rows.append({
        "url":           row["url"],
        "label":         row["label"],
        "category":      row["category"],
        "expected":      row.get("expected_verdict", ""),
        "score":         result["score"],
        "verdict":       result["verdict"],
        "fired_rules":   ", ".join(result["fired_rules"]),
        "findings":      " | ".join(result["findings"]),
    })

url_df = pd.DataFrame(url_rows)
print(f"Done. Verdict distribution:\n{url_df['verdict'].value_counts().to_string()}")
url_df.head(10)

In [ ]:
# Phase 1 metrics
# For binary eval: predict phishing when verdict = 'scam' (strict) or 'scam'|'uncertain' (lenient)

def get_binary_labels(df, verdict_col, threshold="strict"):
    """threshold='strict': only 'scam' counts as phishing prediction.
       threshold='lenient': 'scam' or 'uncertain' counts as phishing."""
    actual = (df["label"] == "phishing").astype(int)
    if threshold == "strict":
        predicted = (df[verdict_col] == "scam").astype(int)
    else:
        predicted = df[verdict_col].isin(["scam", "uncertain"]).astype(int)
    return actual, predicted

for mode in ["strict", "lenient"]:
    actual, predicted = get_binary_labels(url_df, "verdict", mode)
    print(f"\n── Phase 1 URL-Only ({mode} threshold) ─────────────────────────────")
    print(classification_report(actual, predicted,
                                target_names=["legitimate", "phishing"], zero_division=0))

In [ ]:
# Confusion matrix — Phase 1 strict
actual, predicted = get_binary_labels(url_df, "verdict", "strict")
cm = confusion_matrix(actual, predicted)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Legitimate", "Phishing"])
disp.plot(ax=axes[0], colorbar=False, cmap="Blues")
axes[0].set_title("Phase 1: URL-Only (Strict)\nConfusion Matrix")

# Score distribution by label
for label, color in [("legitimate", "steelblue"), ("phishing", "tomato")]:
    subset = url_df[url_df["label"] == label]["score"]
    axes[1].hist(subset, bins=range(0, 12), alpha=0.6, label=label, color=color, edgecolor="white")
axes[1].set_xlabel("Safety Score (0 = scam, 10 = safe)")
axes[1].set_ylabel("Count")
axes[1].set_title("Phase 1: Safety Score Distribution by Label")
axes[1].legend()
axes[1].axvline(7, color="green", linestyle="--", alpha=0.5, label="safe threshold")
axes[1].axvline(4, color="orange", linestyle="--", alpha=0.5, label="uncertain threshold")

plt.tight_layout()
plt.savefig("phase1_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: phase1_results.png")

In [ ]:
# Per-rule breakdown — which rules fire on phishing vs legitimate
all_rules = ["isdomainip", "hasobfuscation", "urlLengthHard", "urlLengthWithComplexity"]

rule_stats = []
for rule in all_rules:
    phishing_rows = url_df[url_df["label"] == "phishing"]
    legit_rows    = url_df[url_df["label"] == "legitimate"]
    phishing_fires = phishing_rows["fired_rules"].str.contains(rule, na=False).sum()
    legit_fires    = legit_rows["fired_rules"].str.contains(rule, na=False).sum()
    rule_stats.append({
        "rule":                 rule,
        "phishing_fires":       phishing_fires,
        "phishing_total":       len(phishing_rows),
        "phishing_fire_rate":   phishing_fires / max(len(phishing_rows), 1),
        "legit_fires":          legit_fires,
        "legit_total":          len(legit_rows),
        "false_positive_rate":  legit_fires / max(len(legit_rows), 1),
    })

rule_df = pd.DataFrame(rule_stats)
print("Per-rule stats (URL heuristics):")
print(rule_df.to_string(index=False))

## 6. Phase 2 — Full Pipeline Evaluation

Fetches each page and runs all three modules. Phishing pages may be offline (especially
if sourced from PhishTank > 24h ago). Falls back to URL-only for unreachable pages.

⏱️ *This cell makes real HTTP requests. Runtime depends on network speed and FETCH_TIMEOUT.*

In [ ]:
print(f"Fetching {len(df)} pages (timeout={FETCH_TIMEOUT}s, workers={MAX_WORKERS})...\n")
page_cache = fetch_all(df["url"].tolist())

fetch_statuses = pd.Series([page_cache[u]["fetch_status"] for u in df["url"]])
ok_count = fetch_statuses.str.startswith("ok").sum()
print(f"\nFetch summary: {ok_count}/{len(df)} pages accessible")
print(fetch_statuses.value_counts().to_string())

In [ ]:
print("Running full pipeline on fetched pages...")
full_rows = []
for _, row in df.iterrows():
    url = row["url"]
    page = page_cache.get(url, {})
    url_result = analyze_url(url)

    if page.get("fetch_status", "").startswith("ok"):
        content_result = analyze_content(page)
        link_result    = analyze_links(page.get("links", []), url)
        fetched = True
    else:
        content_result = EMPTY_CONTENT
        link_result    = EMPTY_LINKS
        fetched = False

    combined = combine_results(url_result, content_result, link_result)
    full_rows.append({
        "url":           url,
        "label":         row["label"],
        "category":      row["category"],
        "fetched":        fetched,
        "fetch_status":  page.get("fetch_status", "not_attempted"),
        "score":         combined["score"],
        "verdict":       combined["verdict"],
        "url_score":     combined["url_score"],
        "content_score": combined["content_score"],
        "link_score":    combined["link_score"],
        "findings_count": len(combined["findings"]),
        "fired_url_rules": ", ".join(combined["fired_url_rules"]),
    })

full_df = pd.DataFrame(full_rows)
print(f"Done. Combined verdict distribution:\n{full_df['verdict'].value_counts().to_string()}")

In [ ]:
# Phase 2 metrics — full pipeline
for mode in ["strict", "lenient"]:
    actual, predicted = get_binary_labels(full_df, "verdict", mode)
    print(f"\n── Phase 2 Full Pipeline ({mode} threshold) ──────────────────────────")
    print(classification_report(actual, predicted,
                                target_names=["legitimate", "phishing"], zero_division=0))

# Also report metrics for URLs where content was actually fetched
fetched_df = full_df[full_df["fetched"] == True]
if len(fetched_df) > 0:
    actual, predicted = get_binary_labels(fetched_df, "verdict", "strict")
    print(f"\n── Phase 2 (fetched pages only, n={len(fetched_df)}, strict) ───────")
    print(classification_report(actual, predicted,
                                target_names=["legitimate", "phishing"], zero_division=0))

In [ ]:
# Phase 2 visualizations
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# (A) Confusion matrix — full pipeline strict
actual, predicted = get_binary_labels(full_df, "verdict", "strict")
cm = confusion_matrix(actual, predicted)
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Legitimate", "Phishing"]).plot(
    ax=axes[0], colorbar=False, cmap="Blues")
axes[0].set_title("Phase 2: Full Pipeline (Strict)\nConfusion Matrix")

# (B) Module score breakdown — phishing vs legitimate
module_cols = ["url_score", "content_score", "link_score"]
phishing_means = full_df[full_df["label"] == "phishing"][module_cols].mean()
legit_means    = full_df[full_df["label"] == "legitimate"][module_cols].mean()
x = np.arange(len(module_cols))
width = 0.35
axes[1].bar(x - width/2, legit_means, width, label="Legitimate", color="steelblue")
axes[1].bar(x + width/2, phishing_means, width, label="Phishing", color="tomato")
axes[1].set_xticks(x)
axes[1].set_xticklabels(["URL Score", "Content Score", "Link Score"])
axes[1].set_ylabel("Mean Safety Score (0=scam, 10=safe)")
axes[1].set_title("Phase 2: Mean Module Score\nby Label")
axes[1].legend()
axes[1].set_ylim(0, 10)

# (C) Combined safety score distribution
for label, color in [("legitimate", "steelblue"), ("phishing", "tomato")]:
    subset = full_df[full_df["label"] == label]["score"]
    axes[2].hist(subset, bins=range(0, 12), alpha=0.6, label=label, color=color, edgecolor="white")
axes[2].set_xlabel("Combined Safety Score")
axes[2].set_ylabel("Count")
axes[2].set_title("Phase 2: Combined Score Distribution")
axes[2].legend()
axes[2].axvline(7, color="green", linestyle="--", alpha=0.5)
axes[2].axvline(4, color="orange", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.savefig("phase2_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: phase2_results.png")

## 7. Summary Table

Side-by-side comparison of all URLs with their expected vs actual verdict.

In [ ]:
# Merge Phase 1 and Phase 2 results for comparison
summary = full_df[["url", "label", "category", "fetched", "score", "verdict",
                   "url_score", "content_score", "link_score", "fired_url_rules"]].copy()
summary["phase1_verdict"] = url_df["verdict"].values
summary["phase1_score"]   = url_df["score"].values
summary["expected"]       = df["expected_verdict"].values

# Flag mismatches
summary["phase2_correct"] = (
    ((summary["label"] == "phishing") & (summary["verdict"].isin(["scam", "uncertain"]))) |
    ((summary["label"] == "legitimate") & (summary["verdict"] == "safe"))
)

# False positives and false negatives
false_positives = summary[(summary["label"] == "legitimate") & (summary["verdict"] == "scam")]
false_negatives = summary[(summary["label"] == "phishing")  & (summary["verdict"] == "safe")]

print(f"False positives (legitimate → scam): {len(false_positives)}")
if len(false_positives):
    print(false_positives[["url", "score", "fired_url_rules"]].to_string(index=False))

print(f"\nFalse negatives (phishing → safe):  {len(false_negatives)}")
if len(false_negatives):
    print(false_negatives[["url", "score", "fired_url_rules"]].to_string(index=False))

summary

## 8. Adding More Data

To expand the dataset with live phishing URLs from PhishTank, run the cell below.
PhishTank URLs go offline within hours — run Phase 2 immediately after downloading.

In [ ]:
# Download a fresh sample of verified phishing URLs from PhishTank.
# Uncomment and run this cell when you want to augment the dataset.

# import io, gzip
# PHISHTANK_URL = "https://data.phishtank.com/data/online-valid.csv.gz"
# print("Downloading PhishTank feed...")
# resp = requests.get(PHISHTANK_URL, timeout=60)
# pt_df = pd.read_csv(io.BytesIO(gzip.decompress(resp.content)))
# sample = pt_df[pt_df["verified"] == "yes"].sample(min(30, len(pt_df)))
#
# new_rows = pd.DataFrame({
#     "url":              sample["url"],
#     "label":            "phishing",
#     "category":         "phishtank-live",
#     "expected_verdict": "scam",
#     "expected_rules":   "",
#     "source":           "phishtank",
#     "notes":            "Community-verified phishing URL from PhishTank daily feed",
# })
# augmented = pd.concat([df, new_rows], ignore_index=True)
# augmented.to_csv(DATASET_PATH, index=False)
# print(f"Dataset expanded to {len(augmented)} rows. Re-run Phase 1 and 2 cells above.")

print("PhishTank import cell — uncomment to run.")